# 09 — Cross-Check & Paper Loop

After the exploration tree settles, two synthesis stages run:

1. **Cross-check** — a reviewer agent examines all findings for contradictions, gaps, and redundancies.
2. **Paper loop** — iterative paper writing with quality threshold and gap→research feedback.

The paper loop is reactive: gaps emit `PaperGapEvent` → watch fires → `DepthRequested` → child team fills the gap → paper rewrites with new findings.

In [ ]:
from lionag2.research.models import (
    CrossCheckReport,
    Contradiction,
    PaperDraft,
    PaperGap,
)

## Cross-check

The cross-checker receives all findings from the Flow and returns a structured `CrossCheckReport`:

```python
checker = Agent(
    "cross_checker",
    prompt=CROSS_CHECK,
    config=self.config,
    response_schema=PromptedSchema(CrossCheckReport),
)
reply = await checker.ask(f"Cross-check {len(findings)} findings:\n{ctx}")
report = await reply.content(retries=1)
```

`PromptedSchema` is needed because AG2 uses `strict: True` with OpenAI, and strict mode requires ALL properties in `required` — but Pydantic excludes fields with `Field(default=...)` from `required`. PromptedSchema bypasses this by injecting the schema into the system prompt instead.

In [ ]:
# CrossCheckReport structure
report = CrossCheckReport(
    contradictions=[
        Contradiction(
            claim_a="AFM glue dominates across all dopings",
            claim_b="Phonon contribution significant at overdoping",
            source_a="theorist d=0",
            source_b="analyst d=1",
            resolution_hint="Restrict AFM claim to optimal doping",
        )
    ],
    gaps=[
        PaperGap(
            section="findings",
            description="No data on underdoped regime",
            research_question="How does pairing symmetry change below optimal doping?",
            priority="high",
        )
    ],
    redundancies=["Multiple branches surveyed the same HgBaCaCuO data"],
    summary="One contradiction found between depth-0 and depth-1 claims.",
)

print(f"Contradictions: {len(report.contradictions)}")
print(f"Gaps: {len(report.gaps)}")
print(f"Summary: {report.summary}")

## Iterative paper loop

The paper writer runs up to `paper_max_iterations` times. Each iteration:

1. Collects all findings + cross-check report.
2. If a previous draft exists, includes its gaps as instructions.
3. Writes a `PaperDraft` with `quality_score`.
4. If quality meets threshold → done.
5. If high-priority gaps exist → emit `PaperGapEvent` per gap.
6. Wait for reactive depth nodes to settle (gap→research→new findings).
7. Refresh findings, loop.

```
paper_draft(quality=0.5) → gap(high) → PaperGapEvent → DepthRequested
                                                              ↓
                                                     child team researches
                                                              ↓
                                                     new findings arrive
                                                              ↓
paper_draft(quality=0.75) → meets threshold → done
```

In [ ]:
# PaperDraft structure
paper = PaperDraft(
    title="Spin-Fluctuation Pairing in Cuprate Superconductors",
    abstract="We investigate the pairing mechanism in layered cuprates...",
    body_markdown="## 1. Introduction\n\nHigh-Tc superconductivity...",
    limitations=["Limited to hole-doped cuprates", "No electron-doped data"],
    gaps=[
        PaperGap(
            section="findings",
            description="Missing underdoped regime data",
            research_question="How does gap symmetry evolve below optimal doping?",
            priority="high",
        ),
    ],
    quality_score=0.65,
)

print(f"Quality: {paper.quality_score}")
print(f"High-priority gaps: {len([g for g in paper.gaps if g.priority == 'high'])}")
print(f"\nRendered length: {len(paper.as_markdown())} chars")
print(paper.as_markdown()[:300])

## Gap → depth expansion feedback

High-priority gaps trigger depth expansion via an observer on the paper writer agent:

```python
@writer.observer(PaperGapEvent)
def _on_gap(event: PaperGapEvent) -> None:
    engine._record(event)
    if event.priority == "high":
        engine._spawn(engine._spawn_depth_node(
            event.research_question, parent_node_id=..., depth=...
        ))
```

This closes the loop: the paper identifies what's missing, the observer spawns a child team to research it, and the paper rewrites with the new evidence.

## Up next

Tutorial 10 covers the recursive depth mechanism — how the engine spawns child teams, deduplicates topics, and manages the exploration tree.